# SummaryEvaluation - Meeting Summary Evaluator
**Course:** CSC 603 - Generative AI | **Team:** Adrian Aquino, Charlie Huynh, Will Brust | **Spring 2026**

## Colab Setup
Run this cell first if you are on Google Colab. It clones the repo and sets up the working directory. Skip if running locally.

In [ ]:
# ── Colab Setup ─────────────────────────────────────────────────────────
# Clones the repo if not already present, then pulls the latest.
# Skips everything when running locally.
import os
import shutil

try:
    from google.colab import files, userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    repo      = 'CSC-603-Capstone---Meeting-Summarizer'
    repo_owner = 'SQ-Ghost'
    os.chdir('/content')

    # guard against nested repo copies left by old buggy runs
    nested_path = os.path.join(repo, repo)
    if os.path.exists(nested_path):
        shutil.rmtree(nested_path)
        print('Cleaned up nested repo copies.')

    # if the repo folder already has a .git dir, just pull; otherwise clone fresh
        print('Repo found. Pulling latest...')
        os.chdir(repo)
    else:
        if os.path.exists(repo):
            shutil.rmtree(repo)
            print('Removed stale repo folder.')
        os.system(f'git clone {repo_url}')
        os.chdir(repo)
        print('Cloned repo.')

    os.system('git pull')

    # configure git identity so commits work on Colab
    print(f'Ready! Working directory: {os.getcwd()}')
else:
    print('Running locally — skipping Colab setup.')


## Install Libraries

In [ ]:
# install all project dependencies
%pip install -q torch transformers accelerate huggingface_hub python-dotenv ipywidgets

print('Done!')

## Import Libraries

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────
import os
import json
import torch                              # GPU / tensor support
from transformers import pipeline         # loads the LLM
from huggingface_hub import login         # authenticates with HuggingFace
from dotenv import load_dotenv, set_key   # reads and writes .env

# if running from the notebooks folder, step up to project root so all paths work
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

# only import colab files if running on Colab
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print('Done!')

## Hugging Face Login

In [ ]:
# ── Hugging Face Login ───────────────────────────────────────────────────
# Colab: reads HF_TOKEN from Secrets panel.
# Local: reads from .env, prompts if missing and saves for next time.
if IN_COLAB:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        print('No HF_TOKEN secret found. Enter your Hugging Face token when prompted, then press Enter.')
        HF_TOKEN = input('HF Token: ')
else:
    load_dotenv()
    HF_TOKEN = os.getenv('HF_TOKEN')

    if not HF_TOKEN:
        print('Enter your Hugging Face token when prompted, then press Enter.')
        HF_TOKEN = input('HF Token: ')
        set_key('.env', 'HF_TOKEN', HF_TOKEN)
        print('Token saved to .env')

login(token=HF_TOKEN)
print('Logged in!')

## Load the Model

In [ ]:
# ── Model Setup ──────────────────────────────────────────────────────────
# Uses GPU (float16) when available for speed; falls back to CPU (float32).
MODEL_NAME = 'meta-llama/Llama-3.2-1B-Instruct'

# use GPU if available, otherwise CPU
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using: {DEVICE}')

llm = pipeline(
    task='text-generation',
    model=MODEL_NAME,
    device_map='auto',
    torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32
)

print('Model loaded!')

## Summarize Function

In [ ]:
# ── Load Prompts from RecapAI ────────────────────────────────────────────
# Always pulls ROLE and TASK from RecapAI.ipynb so both notebooks stay in sync.
# read ROLE and TASK directly from RecapAI.ipynb so we always use the latest prompts
with open('notebooks/RecapAI.ipynb', 'r', encoding='utf-8') as f:
    _nb = json.load(f)

ROLE = None
TASK = None
for _cell in _nb['cells']:
    _source = ''.join(_cell['source'])
    if 'ROLE = "' in _source and 'TASK = """' in _source:
        role_start = _source.index('ROLE = "') + 8
        role_end   = _source.index('"', role_start)
        ROLE       = _source[role_start:role_end]
        task_start = _source.index('TASK = """') + 10
        task_end   = _source.index('"""', task_start)
        TASK       = _source[task_start:task_end]
        break

if ROLE and TASK:
    print('ROLE and TASK loaded from RecapAI.ipynb.')
else:
    print('WARNING: Could not find ROLE/TASK in RecapAI.ipynb.')


def summarize_transcript(transcript, max_tokens=1024):
    """Sends a transcript to the model and returns a parsed JSON summary.

    :param transcript: The meeting transcript text.
    :type transcript: str
    :param max_tokens: Max new tokens the model can generate.
    :type max_tokens: int
    :returns: JSON with summary, decisions, assigned_tasks, open_questions.
    :rtype: dict
    """
    messages = [
        {'role': 'system', 'content': ROLE},
        {'role': 'user',   'content': f'{TASK}\n\nTranscript:\n{transcript}'}
    ]
    response = llm(messages, max_new_tokens=max_tokens, do_sample=False, temperature=1.0)
    raw      = response[0]['generated_text'][-1]['content']

    try:
        start  = raw.index('{')
        end    = raw.rindex('}') + 1
        return json.loads(raw[start:end])
    except (ValueError, json.JSONDecodeError):
        return {'summary': raw, 'decisions': [], 'assigned_tasks': [], 'open_questions': []}


print('Function defined!')

## Load Summaries
Reads pre-computed summaries from the `Summaries/` folder produced by RecapAI. Real-world transcripts are summarized here on the fly.

In [ ]:
# scan for pre-computed mock summaries in the Summaries folder
mock_summary_files = []

if os.path.exists('Summaries'):
    mock_summary_files = sorted([
        f for f in os.listdir('Summaries')
        if f.startswith('GeneratedMockTranscript-') and f.endswith('_summary.json')
    ])

if mock_summary_files:
    print(f'Found {len(mock_summary_files)} mock summary file(s) in Summaries/:')
    for f in mock_summary_files:
        print(f'  {f}')
else:
    print('No mock summaries found in Summaries/.')
    print('Run option 3 in RecapAI first to generate and summarize transcripts.')

## Load Real-World Transcripts
Supports QMSum and TCR JSON formats. Place `.json` files in this folder to evaluate against real meetings.

In [ ]:
# ── Real-World Transcript Loaders ────────────────────────────────────────
# Supports two formats: QMSum and TCR. detect_and_load picks the right one.
# QMSum format: {"meeting_transcripts": [{"speaker": "...", "content": "..."}]}
def load_qmsum(filepath):
    """Loads a QMSum-format transcript JSON and returns plain text.

    :param filepath: Path to the QMSum JSON file.
    :type filepath: str
    :returns: Transcript as a single string, one line per speaker turn.
    :rtype: str
    """
    with open(filepath, 'r') as f:
        data = json.load(f)
    turns = data.get('meeting_transcripts', [])
    return '\n'.join(f"{t['speaker']}: {t['content']}" for t in turns)


# TCR format: deeply nested — data_source > meeting_name > topics > transcripts
def load_tcr(filepath):
    """Loads a TCR-format transcript JSON and returns plain text.

    :param filepath: Path to the TCR JSON file.
    :type filepath: str
    :returns: Transcript as a single string, one line per speaker turn.
    :rtype: str
    """
    with open(filepath, 'r') as f:
        data = json.load(f)
    lines = []
    for source in data.values():
        for meeting in source.values():
            for topic in meeting.get('topics', {}).values():
                for turn in topic.get('transcripts', []):
                    lines.append(f"{turn.get('speaker', 'Speaker')}: {turn.get('contents', '')}")
    return '\n'.join(lines)


# picks the right loader by checking for QMSum's top-level key
def detect_and_load(filepath):
    """Auto-detects QMSum vs TCR format and loads the transcript.

    :param filepath: Path to the transcript JSON file.
    :type filepath: str
    :returns: Transcript as plain text.
    :rtype: str
    """
    with open(filepath, 'r') as f:
        data = json.load(f)
    if 'meeting_transcripts' in data:
        return load_qmsum(filepath)
    return load_tcr(filepath)


# scan for real-world JSON files in the RealWorldTranscripts folder
os.makedirs('RealWorldTranscripts', exist_ok=True)
realworld_files = sorted([
    f for f in os.listdir('RealWorldTranscripts')
    if f.endswith('.json')
])

if realworld_files:
    print(f'Found {len(realworld_files)} real-world transcript file(s):')
    for f in realworld_files:
        print(f'  {f}')
else:
    print('No real-world transcript JSON files found.')
    print('Download QMSum or TCR .json files and place them in the RealWorldTranscripts folder.')

print('Functions defined!')

## Evaluation Prompt and Functions

In [ ]:
# ── Evaluation Prompt ────────────────────────────────────────────────────
# The model scores the summary against the original transcript
# and returns JSON with scores, issues, and prompt improvement suggestions.
EVAL_PROMPT = """Score this AI meeting summary against the original transcript.

Return ONLY valid JSON with these keys:
- scores: summary (1-5), decisions (1-5), assigned_tasks (1-5), open_questions (1-5), overall (1-10)
- issues: list of errors found
- prompt_suggestions: list of fixes to improve the prompt

Scores: 5 = correct and complete, 1 = wrong or missing

Watch for:
- Tasks assigned to the wrong speaker
- Facts or statements listed as decisions
- Resolved items listed as open questions
- Deadlines not mentioned in the transcript
- Duplicate or missing items

For suggestions, be direct. Example:
Add to prompt: Only assign a task to someone who explicitly volunteered.
"""


def evaluate(transcript, summary):
    """Sends transcript + summary to the model and returns a scored evaluation.

    :param transcript: The original meeting transcript.
    :type transcript: str
    :param summary: The JSON summary output from RecapAI.
    :type summary: dict
    :returns: Dict with scores, issues, and prompt_suggestions.
    :rtype: dict
    """
    messages = [
        {'role': 'system', 'content': EVAL_PROMPT},
        {'role': 'user',   'content': f'Transcript:\n{transcript}\n\nSummary:\n{json.dumps(summary, indent=2)}'}
    ]

    print('Sending to model for evaluation...')
    response = llm(messages, max_new_tokens=1024, do_sample=False, temperature=1.0)
    raw = response[0]['generated_text'][-1]['content']
    print('Response received. Parsing...')

    return try_parse_eval(raw)


def try_parse_eval(raw):
    """Parses evaluation JSON from the model output. Falls back to raw text on failure.

    :param raw: Raw text from the model.
    :type raw: str
    :returns: Parsed evaluation dict or fallback with raw text.
    :rtype: dict
    """
    try:
        start  = raw.index('{')
        end    = raw.rindex('}') + 1
        parsed = json.loads(raw[start:end])
        print('Parsed successfully.')
        return parsed
    except (ValueError, json.JSONDecodeError):
        print('Could not parse output. Returning raw text.')
        return {'raw': raw}


def display_results(result):
    """Prints scores, issues, and prompt suggestions to the cell output.

    :param result: Parsed evaluation output from evaluate().
    :type result: dict
    """
    if 'raw' in result:
        print('Raw output (could not parse):')
        print(result['raw'])
        return

    scores = result.get('scores', {})
    print('=' * 40)
    print('EVALUATION SCORES')
    print('=' * 40)
    print('  Summary:        ' + str(scores.get('summary', 'N/A')) + ' / 5')
    print('  Decisions:      ' + str(scores.get('decisions', 'N/A')) + ' / 5')
    print('  Assigned Tasks: ' + str(scores.get('assigned_tasks', 'N/A')) + ' / 5')
    print('  Open Questions: ' + str(scores.get('open_questions', 'N/A')) + ' / 5')
    print('  Overall:        ' + str(scores.get('overall', 'N/A')) + ' / 10')
    print()

    issues = result.get('issues', [])
    if issues:
        print('ISSUES FOUND')
        print('-' * 40)
        for i, issue in enumerate(issues, 1):
            print('  ' + str(i) + '. ' + issue)
        print()

    suggestions = result.get('prompt_suggestions', [])
    if suggestions:
        print('PROMPT IMPROVEMENT SUGGESTIONS')
        print('-' * 40)
        for i, s in enumerate(suggestions, 1):
            print('  ' + str(i) + '. ' + s)
        print()


print('Functions defined!')

## Run — Evaluate All Transcripts

In [ ]:
# ── Evaluate All Transcripts ─────────────────────────────────────────────
# Runs evaluate() on every mock and real-world transcript,
# saves results to Evaluations/ and a timestamped EvaluationRuns/ folder.
# build list of (label, transcript_text, summary_dict) to evaluate
all_items = []

# load mock summaries from Summaries/ and pair with their original transcripts
for filename in mock_summary_files:
    transcript_name = filename.replace("_summary.json", ".txt")
    transcript_path = f"GeneratedMockTranscripts/{transcript_name}"
    if os.path.exists(transcript_path):
        with open(transcript_path, "r") as f:
            transcript = f.read()
        with open(f"Summaries/{filename}", "r") as f:
            summary = json.load(f)
        all_items.append((filename.replace("_summary.json", ""), transcript, summary))
    else:
        print(f"Skipping {filename}: transcript not found at {transcript_path}")

# real-world transcripts: summarize them here and save to Summaries/
os.makedirs("Summaries", exist_ok=True)
for filename in realworld_files:
    try:
        transcript = detect_and_load(f"RealWorldTranscripts/{filename}")
        print(f"\nSummarizing {filename}...")
        summary = summarize_transcript(transcript)
        out_name = filename.replace(".json", "_summary.json")
        with open(f"Summaries/{out_name}", "w") as f:
            json.dump(summary, f, indent=2)
        all_items.append((filename.replace(".json", ""), transcript, summary))
    except Exception as e:
        print(f"Skipping {filename}: {e}")

if not all_items:
    print("No items to evaluate.")
    print("Run RecapAI option 3 to generate transcripts, or add real-world JSON files to RealWorldTranscripts/.")
else:
    os.makedirs("Evaluations", exist_ok=True)
    os.makedirs("EvaluationRuns", exist_ok=True)

    # each run gets its own folder so history is never overwritten
    from datetime import datetime
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    existing_runs = [d for d in os.listdir("EvaluationRuns") if os.path.isdir(f"EvaluationRuns/{d}")]
    run_number = len(existing_runs) + 1
    run_folder = f"EvaluationRuns/run_{run_number:03d}_{timestamp}"
    os.makedirs(run_folder, exist_ok=True)
    print(f"\nRun #{run_number} — saving history to {run_folder}")

    all_scores = []

    for label, transcript, summary in all_items:
        print(f"\n{chr(61) * 40}")
        print(f"Evaluating: {label}")
        print(chr(61) * 40)

        result = evaluate(transcript, summary)
        display_results(result)

        eval_data = {"file": label, "summary": summary, "evaluation": result}
        out_name = label + "_evaluation.json"

        # Evaluations/ is read by the auto-replace cell below
        with open(f"Evaluations/{out_name}", "w") as f:
            json.dump(eval_data, f, indent=2)

        # run folder keeps a permanent snapshot of this run
        with open(f"{run_folder}/{out_name}", "w") as f:
            json.dump(eval_data, f, indent=2)

        print(f"Saved: Evaluations/{out_name}")

        if "scores" in result:
            all_scores.append({"file": label, **result["scores"]})

    if all_scores:
        print("\n" + chr(61) * 50)
        print("OVERALL RESULTS")
        print(chr(61) * 50)
        print(f"{'File':<35} {'Sum':>4} {'Dec':>4} {'Tasks':>6} {'OQ':>4} {'Total':>6}")
        print("-" * 50)
        for s in all_scores:
            print(f"{s['file']:<35} {str(s.get('summary','?')):>4} {str(s.get('decisions','?')):>4} {str(s.get('assigned_tasks','?')):>6} {str(s.get('open_questions','?')):>4} {str(s.get('overall','?')):>6}")

        # average each score dimension across all evaluated items
        score_keys = ["summary", "decisions", "assigned_tasks", "open_questions", "overall"]
        averages = {}
        for k in score_keys:
            vals = [s[k] for s in all_scores if k in s and isinstance(s[k], (int, float))]
            averages[k] = round(sum(vals) / len(vals), 2) if vals else None

        print("\nAverages:")
        for k, v in averages.items():
            print(f"  {k}: {v}")

        # one JSON file per run — scores, prompts used, timestamp, averages
        run_summary = {
            "run": run_number,
            "timestamp": timestamp,
            "prompt_role": ROLE,
            "prompt_task": TASK,
            "items_evaluated": len(all_items),
            "scores": all_scores,
            "averages": averages,
        }
        with open(f"{run_folder}/run_summary.json", "w") as f:
            json.dump(run_summary, f, indent=2)
        print(f"\nRun summary saved to {run_folder}/run_summary.json")

        # compare this run's averages to the previous run to show improvement
        all_run_dirs = sorted([d for d in os.listdir("EvaluationRuns") if os.path.isdir(f"EvaluationRuns/{d}")])
        prev_runs = [r for r in all_run_dirs if r != os.path.basename(run_folder)]
        if prev_runs:
            prev_path = f"EvaluationRuns/{prev_runs[-1]}/run_summary.json"
            if os.path.exists(prev_path):
                with open(prev_path, "r") as f:
                    prev_summary = json.load(f)
                prev_avg = prev_summary.get("averages", {})
                print("\nScore change vs previous run:")
                change_map = {}
                for k in score_keys:
                    curr_v = averages.get(k)
                    prev_v = prev_avg.get(k)
                    if curr_v is not None and prev_v is not None:
                        delta = round(curr_v - prev_v, 2)
                        change_map[k] = delta
                        print(f"  {k}: {prev_v} -> {curr_v}  ({'+' if delta >= 0 else ''}{delta})")
                run_summary["score_change"] = change_map
                with open(f"{run_folder}/run_summary.json", "w") as f:
                    json.dump(run_summary, f, indent=2)

    print(f"\nDone! Evaluated {len(all_items)} item(s).")

## Generate Improved Prompts
Reads all evaluation results and generates updated ROLE and TASK prompts you can copy into RecapAI.

In [ ]:
# ── Improve Prompts ──────────────────────────────────────────────────────
# Reads all evaluation suggestion lists, sends them to the LLM to generate
# an improved ROLE and TASK, then updates RecapAI.ipynb and backend.py.
# Also re-summarizes all transcripts with the new prompts.
# collect all prompt suggestions from saved evaluation files in the Evaluations folder
all_suggestions = []
eval_files = sorted([f for f in os.listdir('Evaluations') if f.endswith('_evaluation.json')]) if os.path.exists('Evaluations') else []

for filename in eval_files:
    with open(f'Evaluations/{filename}', 'r') as f:
        data = json.load(f)
    suggestions = data.get('evaluation', {}).get('prompt_suggestions', [])
    all_suggestions.extend(suggestions)

if not all_suggestions:
    print('No suggestions found. Run the evaluation cells first.')
else:
    print(f'Collected {len(all_suggestions)} suggestion(s) from {len(eval_files)} evaluation file(s).')

    suggestions_text = '\n'.join(f'- {s}' for s in all_suggestions)

    improve_prompt = (
        'You are improving an AI prompt based on evaluation feedback.\n\n'
        'Current ROLE:\n' + ROLE + '\n\n'
        'Current TASK:\n' + TASK + '\n\n'
        'Suggestions from evaluation:\n' + suggestions_text + '\n\n'
        'Return only two clearly labeled blocks:\n'
        'ROLE: <improved role>\n'
        'TASK: <improved task>\n\n'
        'Do not add any other text.'
    )

    messages = [
        {'role': 'system', 'content': 'You improve AI prompts based on evaluation feedback.'},
        {'role': 'user',   'content': improve_prompt}
    ]

    print('Generating improved prompts...')
    response = llm(messages, max_new_tokens=512, do_sample=False, temperature=1.0)
    raw = response[0]['generated_text'][-1]['content']

    print('\n' + '=' * 50)
    print('GENERATED PROMPTS:')
    print('=' * 50)
    print(raw)

    # extract ROLE: and TASK: blocks from the model's plain-text reply
    new_role = None
    new_task = None
    lines = raw.strip().split('\n')
    for i, line in enumerate(lines):
        if line.startswith('ROLE:') and new_role is None:
            new_role = line[5:].strip()
        elif line.startswith('TASK:') and new_task is None:
            task_lines = [line[5:].strip()] + lines[i + 1:]
            new_task = '\n'.join(task_lines).strip()
            break

    if not new_role or not new_task:
        print('\nCould not parse ROLE or TASK. Copy manually from above.')
    else:
        print('\nReplace the prompts in RecapAI.ipynb and backend.py with the new ones?')
        choice = input('Enter yes or no: ').strip().lower()

        if choice == 'yes':
            # locate the prompt cell by content, not cell ID, so it works after edits
            with open('notebooks/RecapAI.ipynb', 'r', encoding='utf-8') as f:
                nb = json.load(f)

            for cell in nb['cells']:
                source = ''.join(cell['source'])
                if 'ROLE = "' in source and 'TASK = """' in source:
                    role_start = source.index('ROLE = "')
                    role_end   = source.index('"', role_start + 8) + 1
                    source     = source[:role_start] + f'ROLE = "{new_role}"' + source[role_end:]

                    task_start = source.index('TASK = """')
                    task_end   = source.index('"""', task_start + 10) + 3
                    source     = source[:task_start] + f'TASK = """{new_task}"""' + source[task_end:]

                    cell['source'] = source.splitlines(True)
                    break

            with open('notebooks/RecapAI.ipynb', 'w', encoding='utf-8') as f:
                json.dump(nb, f, indent=1, ensure_ascii=False)

            print('RecapAI.ipynb updated.')

            # mirror the same prompt change into the Gradio app's backend
            with open('app/backend.py', 'r', encoding='utf-8') as f:
                backend_src = f.read()

            prompt_marker = 'SYSTEM_PROMPT = """'
            ps            = backend_src.index(prompt_marker)
            pe            = backend_src.index('"""', ps + len(prompt_marker)) + 3
            new_block     = 'SYSTEM_PROMPT = """\\\n' + new_role + '\n\n' + new_task + '\n"""'
            backend_src   = backend_src[:ps] + new_block + backend_src[pe:]

            with open('app/backend.py', 'w', encoding='utf-8') as f:
                f.write(backend_src)

            print('backend.py updated.')
            print('Done! Both files have been updated with the new prompts.')

            # ask the LLM to explain what changed in plain English
            print('\nGenerating improvement explanation...')
            explain_prompt = '\n'.join([
                'You just improved an AI prompt based on evaluation feedback. '
                'In 2-3 sentences, explain what specific changes you made to the ROLE and TASK prompts and why they should improve summary quality.',
                '',
                'Old ROLE: ' + ROLE,
                'New ROLE: ' + new_role,
                '',
                'Old TASK:',
                TASK,
                '',
                'New TASK:',
                new_task,
                '',
                'Suggestions used:',
                suggestions_text,
            ])
            explain_resp = llm([{'role': 'user', 'content': explain_prompt}], max_new_tokens=200, do_sample=False, temperature=1.0)
            explanation = explain_resp[0]['generated_text'][-1]['content'].strip()
            print('\nImprovement Explanation:')
            print(explanation)

            # append improvement notes to the current run's history folder
            from datetime import datetime as _dt
            if os.path.exists('EvaluationRuns'):
                run_dirs = sorted([d for d in os.listdir('EvaluationRuns') if os.path.isdir(f'EvaluationRuns/{d}')])
                if run_dirs:
                    latest_run = f'EvaluationRuns/{run_dirs[-1]}'
                    improvement_data = {
                        'timestamp': _dt.now().strftime('%Y-%m-%d_%H-%M-%S'),
                        'old_role': ROLE,
                        'old_task': TASK,
                        'new_role': new_role,
                        'new_task': new_task,
                        'suggestions_used': all_suggestions,
                        'explanation': explanation,
                    }
                    with open(f'{latest_run}/improvement_notes.json', 'w') as f:
                        json.dump(improvement_data, f, indent=2)
                    print(f'Improvement notes saved to {latest_run}/improvement_notes.json')

            # update session globals so re-summarization below uses the new prompts
            ROLE = new_role
            TASK = new_task
            print('\nRe-summarizing all transcripts with improved prompts...')

            # re-summarize all mock transcripts with the improved prompts
            mock_txts = sorted([f for f in os.listdir('GeneratedMockTranscripts') if f.endswith('.txt')]) if os.path.exists('GeneratedMockTranscripts') else []
            for txt_file in mock_txts:
                try:
                    with open(f'GeneratedMockTranscripts/{txt_file}', 'r') as f:
                        transcript = f.read()
                    print(f'  Re-summarizing {txt_file}...')
                    new_summary = summarize_transcript(transcript)
                    out_name = txt_file.replace('.txt', '_summary.json')
                    with open(f'Summaries/{out_name}', 'w') as f:
                        json.dump(new_summary, f, indent=2)
                except Exception as e:
                    print(f'  Skipping {txt_file}: {e}')

            # re-summarize all real-world transcripts with the improved prompts
            rw_files = sorted([f for f in os.listdir('RealWorldTranscripts') if f.endswith('.json')]) if os.path.exists('RealWorldTranscripts') else []
            for rw_file in rw_files:
                try:
                    transcript = detect_and_load(f'RealWorldTranscripts/{rw_file}')
                    print(f'  Re-summarizing {rw_file}...')
                    new_summary = summarize_transcript(transcript)
                    out_name = rw_file.replace('.json', '_summary.json')
                    with open(f'Summaries/{out_name}', 'w') as f:
                        json.dump(new_summary, f, indent=2)
                except Exception as e:
                    print(f'  Skipping {rw_file}: {e}')

            print('All summaries updated with improved prompts.')

            # also improve the mock transcript generation prompt
            current_mock_prompt = ''
            with open('notebooks/RecapAI.ipynb', 'r', encoding='utf-8') as f:
                _recap = json.load(f)
            for _cell in _recap['cells']:
                _src = ''.join(_cell['source'])
                if 'MOCK_PROMPT = """' in _src:
                    _ms = _src.index('MOCK_PROMPT = """') + 16
                    _me = _src.index('"""', _ms)
                    current_mock_prompt = _src[_ms:_me]
                    break

            # use the same evaluation feedback to tighten the mock transcript prompt
            print('\nImproving mock transcript generation prompt...')
            mock_improve_prompt = '\n'.join([
                'You are improving a prompt used to generate realistic mock meeting transcripts.',
                'The current prompt sometimes produces transcripts with formatting artifacts',
                '(e.g. quotation marks around speech, stage directions, bullet points, structured formatting).',
                '',
                'Current MOCK_PROMPT:',
                current_mock_prompt,
                '',
                'Issues found during evaluation:',
                suggestions_text,
                '',
                'Return ONLY the improved prompt text with no extra commentary.',
            ])
            mock_resp = llm([{'role': 'user', 'content': mock_improve_prompt}], max_new_tokens=300, do_sample=False, temperature=1.0)
            new_mock_prompt = mock_resp[0]['generated_text'][-1]['content'].strip()
            print('\nImproved MOCK_PROMPT:')
            print(new_mock_prompt)

            # write the improved mock prompt back into the notebook
            with open('notebooks/RecapAI.ipynb', 'r', encoding='utf-8') as f:
                recap_nb = json.load(f)
            for cell in recap_nb['cells']:
                cell_src = ''.join(cell['source'])
                if 'MOCK_PROMPT = """' in cell_src:
                    mp_start = cell_src.index('MOCK_PROMPT = """')
                    mp_end   = cell_src.index('"""', mp_start + 16) + 3
                    cell['source'] = cell_src[:mp_start] + 'MOCK_PROMPT = """' + new_mock_prompt + '"""' + cell_src[mp_end:]
                    break
            with open('notebooks/RecapAI.ipynb', 'w', encoding='utf-8') as f:
                json.dump(recap_nb, f, indent=1, ensure_ascii=False)
            print('MOCK_PROMPT updated in RecapAI.ipynb.')
            print('Cancelled. Copy the prompts manually from above if needed.')